In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

from tarifalatam.config import FX_RATES, PROCESSED_DIR, REPORTS_DIR
from tarifalatam.data.pipeline import build_datasets
from tarifalatam.data.split import assert_disjoint_hosts
from tarifalatam.models.baseline import baseline_cv_mae

# 03 · Preprocesamiento — de las decisiones del EDA a los conjuntos de entrenamiento y prueba

Este notebook **solo orquesta**. Las reglas viven en `src/tarifalatam/data/cleaning.py` (una función por decisión del EDA,
cada una con su prueba en `tests/`), la partición en `split.py` y el orden de los pasos en `pipeline.py`
(ver `specs/001-eda-limpieza-particion/research.md`, §4).

**Entradas:** los siete archivos de `data/raw/` (no se modifican); Buenos Aires se excluye en el primer paso
(decisión del grupo del 2026-10-31). **Salidas:** `data/processed/train.csv.gz`,
`data/processed/test.csv.gz` y `reports/03_exclusiones.csv`.

> **La prueba está cerrada.** De `test` solo se cuentan filas y anfitriones. No se calculan promedios, gráficos ni
> métricas sobre ella hasta la etapa de validación.

## 0. Tasas de cambio

Los precios se pasan a USD con la tasa oficial del banco central de cada país en la fecha del corte (`FX_RATES` en
`src/tarifalatam/config.py`). Si falta alguna, `build_datasets()` se detiene en la sección 1 con el mensaje
"falta la tasa de cambio de …": **nunca se usa una tasa supuesta**.

In [2]:
# Solo se muestran: la validación (que exista, que sea positiva y que la moneda coincida) la hace to_usd() en src/
pd.DataFrame(FX_RATES).T[["currency", "local_per_usd", "date", "source"]]

,currency,local_per_usd,date,source
rio-de-janeiro,BRL,5.1161,2026-09-22,"Banco Central do Brasil, PTAX venta del 2026-0..."
sao-paulo,BRL,5.1696,2026-09-14,"Banco Central do Brasil, PTAX venta del 2026-0..."
mexico-city,MXN,17.3015,2026-09-23,"Banxico, tipo de cambio FIX publicado en el DO..."
buenos-aires,ARS,1516.2964,2026-09-23,"BCRA, tipo de cambio de referencia Com. A 3500..."
bogota,COP,3072.27,2026-09-14,"Superfinanciera, TRM vigente del 2026-09-12 al..."
santiago,CLP,945.87,2026-09-23,"Banco Central de Chile, dólar observado al 202..."
belize,USD,1.0,2026-09-25,Inside Airbnb publica Belice en USD (price_quo...


## 1. Limpiar y partir por anfitrión

`build_datasets()` aplica las reglas que no aprenden de los datos, parte por `host_id` y, al final, aprende los límites de
precio por ciudad **solo con el entrenamiento**.

In [3]:
train, test, exclusions, bounds = build_datasets()
assert_disjoint_hosts(train, test)  # ningún anfitrión en los dos conjuntos

sizes = pd.DataFrame(
    {
        "filas train": train.groupby("city").size(),
        "filas test": test.groupby("city").size(),
        "anfitriones train": train.groupby("city")["host_id"].nunique(),
        "anfitriones test": test.groupby("city")["host_id"].nunique(),
    }
)
print(f"entrenamiento: {len(train):,} filas, {train['host_id'].nunique():,} anfitriones")
print(f"prueba:        {len(test):,} filas, {test['host_id'].nunique():,} anfitriones "
      f"({len(test) / (len(train) + len(test)):.1%} de las filas)")
print("anfitriones en común:", len(set(train["host_id"]) & set(test["host_id"])))
sizes

entrenamiento: 118,513 filas, 53,385 anfitriones
prueba:        29,721 filas, 13,345 anfitriones (20.1% de las filas)
anfitriones en común: 0


,filas train,filas test,anfitriones train,anfitriones test
city,,,,
belize,2604,853,773,226
bogota,12205,3552,4777,1145
mexico-city,22291,5388,9520,2401
rio-de-janeiro,34091,8246,19183,4774
santiago,14128,3399,7261,1834
sao-paulo,33194,8283,12039,2996


## 2. Registro de exclusiones: cuántas filas quitó cada regla

In [4]:
counts = exclusions[exclusions["step"] != "price_bounds"]
summary = counts.pivot_table(index="city", columns=["split", "step"], values="removed", aggfunc="sum", sort=False)
summary.insert(0, ("", "cargadas"), counts[counts["step"] == "excluded_city"].set_index("city")["before"])
summary[("", "finales")] = sizes["filas train"] + sizes["filas test"]
summary.loc["total"] = summary.sum()
assert (counts["before"] - counts["removed"] == counts["after"]).all(), "el registro no cuadra"
summary

split                              all                                                      train          test          
step            cargadas excluded_city previous_scrape missing_price monthly_rental price_outlier price_outlier   finales
city                                                                                                                     
belize            3671.0           0.0           135.0          49.0            6.0          19.0           5.0    3457.0
bogota           19505.0           0.0           318.0         518.0         2873.0          35.0           4.0   15757.0
buenos-aires     29753.0       29753.0            <NA>          <NA>           <NA>          <NA>          <NA>      <NA>
mexico-city      30468.0           0.0          2184.0         380.0          153.0          56.0          16.0   27679.0
rio-de-janeiro   48247.0           0.0          4811.0         573.0          190.0         290.0          46.0   42337.0
santiago         18904.0           0.0           863.0         310.0          169.0          29.0           6.0   17527.0
sao-paulo        44080.0           0.0          1277.0         705.0          557.0          55.0           9.0   41477.0
total           194628.0       29753.0          9588.0        2535.0         3948.0         484.0          86.0  148234.0

In [5]:
# Límites de precio aprendidos SOLO con el entrenamiento: [0,1 x, 20 x] la mediana de cada ciudad, en USD
exclusions[exclusions["step"] == "price_bounds"].set_index("city")[["low_usd", "high_usd"]].round(1)

,low_usd,high_usd
city,,
belize,33.4,6680.0
bogota,5.5,1109.1
mexico-city,8.6,1725.3
rio-de-janeiro,8.8,1759.2
santiago,5.9,1173.5
sao-paulo,6.6,1319.3


## 3. Guardar los conjuntos y el registro

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_csv(PROCESSED_DIR / "train.csv.gz", index=False, compression="gzip")
test.to_csv(PROCESSED_DIR / "test.csv.gz", index=False, compression="gzip")
exclusions.to_csv(REPORTS_DIR / "03_exclusiones.csv", index=False)
for path in (PROCESSED_DIR / "train.csv.gz", PROCESSED_DIR / "test.csv.gz", REPORTS_DIR / "03_exclusiones.csv"):
    print(f"{path.relative_to(PROCESSED_DIR.parents[1])}: {path.stat().st_size / 1e6:.1f} MB")

data\processed\train.csv.gz: 18.2 MB
data\processed\test.csv.gz: 4.5 MB
reports\03_exclusiones.csv: 0.0 MB


## 4. Línea base (solo con el entrenamiento)

Mediana de `price_usd` por ciudad × `room_type`, evaluada con 5 pliegues por anfitrión dentro del entrenamiento. Es el MAE
que el modelo de la etapa 4 tiene que bajar.

In [7]:
baseline = baseline_cv_mae(train, n_splits=5)
print("MAE por pliegue (USD):", [round(m, 2) for m in baseline["per_fold"]])
print(f"MAE de la línea base: {baseline['mean']:.2f} ± {baseline['std']:.2f} USD por noche")

MAE por pliegue (USD): [57.95, 54.45, 54.77, 58.64, 53.99]
MAE de la línea base: 55.96 ± 1.94 USD por noche


## 5. Qué copiar al informe

- El MAE de la línea base → `reports/01_eda.md` ("Qué son los datos" → "Línea base").
- Las filas quitadas por regla → `especificaciones/03-preprocesamiento.md`, sección 3.
- Comprobar las pruebas desde la raíz del repo: `pytest`.